<div style="display: flex; gap: 10px;">
  <img src="../images/HOOPS_AI.jpg" style="width: 20%;">
</div>

# Quickstart: turn a CAD shape into expert knowledge

## What would an expert say about this part?

<img src="../images/context_layer_intro.png" style="width: 100%;">

Show an experienced manufacturing engineer the 3D model of this gear, and within seconds they can
tell you:

- **Sourcing options** — which suppliers and processes can make it
- **Cost drivers** — what makes it cheap or expensive
- **Manufacturing process** — how it is actually produced, step by step
- **Tolerance sensitivity** — where precision matters most

That judgement comes from years of experience — and today it lives in just a few experts' heads,
so it does not scale.

**This notebook shows how to move that expert knowledge into a tool anyone can use**, by
integrating the HOOPS AI Context Layer into your own systems. Starting from the shape alone, it
recovers the same kind of business knowledge automatically.

> **Turn 3D shapes into actionable business knowledge.**


## How it works: the big picture

The Context Layer turns a shape into business knowledge in three steps:

1. **Find** parts that look like your part (CAD similarity search).
2. **Read** the metadata you already store for those look-alike parts.
3. **Predict** the missing fields for your part — each with a confidence score.

<img src="../images/from_geometry_to_intelligence.png" style="width: 100%;">

On the left, similar gears are retrieved from the shape alone, each with a similarity score. On
the right, the metadata those parts share becomes the business insight you care about — how the
part is made, what drives its cost, and where it can be sourced.

The rest of this notebook runs those three steps on a single gear, with the smallest amount of
code. To customize the behaviour — combining rules, handling messy data, tuning confidence,
estimating cost, and auditing labels — see the companion notebook
[`demo_HOOPS_Embeddings_Context_Layer_advanced.ipynb`](./demo_HOOPS_Embeddings_Context_Layer_advanced.ipynb).

**Before you start:** this notebook loads a pre-built search index created by
[`demo_HOOPS_Embeddings_indexing.ipynb`](./demo_HOOPS_Embeddings_indexing.ipynb). Run that
notebook first if you haven't already.


In [ ]:
import hoops_ai
import os
import sys

license_key = os.environ.get("HOOPS_AI_LICENSE")
if not license_key:
    sys.exit("HOOPS_AI_LICENSE environment variable is required.")

hoops_ai.set_license(license_key, validate=True)


## 1. Load the model and the search index

We load the pre-trained embedding model, open the FAISS index built during indexing, and point
to a folder of example query parts.


In [ ]:
import pathlib
from hoops_ai.ml.embeddings import HOOPSEmbeddings
from hoops_ai.ml import CADSearch
from hoops_ai.storage import CADFileRetriever, LocalStorageProvider
from hoops_ai.insights import DatasetViewer

tmcad = pathlib.Path.cwd().parent.joinpath("packages", "vectorstores", "tmcad")

HOOPSEmbeddings.register_model(
    model_name="HOOPS Embeddings SIGNAL preview",
    checkpoint_path=str(pathlib.Path.cwd().parent.joinpath(
        "packages", "trained_ml_models", "ts3d_2M_hoops_embeddings_SIGNAL-preview.ckpt")),
)
embedder = HOOPSEmbeddings(model="HOOPS Embeddings SIGNAL preview")

searcher = CADSearch(shape_model=embedder)
searcher.load_shape_index(path=str(tmcad.joinpath("TMCAD_SIGNAL.faiss")))

retriever = CADFileRetriever(
    storage_provider=LocalStorageProvider(directory_path=tmcad.joinpath("queries_for_demo")),
    formats=[".stp", ".step", ".iges", ".igs"],
)
cad_files = retriever.get_file_list()
viewer = DatasetViewer([], [], [], reference_dir=tmcad.joinpath("images_tmcad"))
print(f"Ready. {len(cad_files)} example query parts available.")


## 2. Connect your metadata store

The Context Layer reads metadata through a `ContextProvider`. In your project this is a small
class that talks to your PLM, ERP, or database. Here we use `OnDemandContextProvider` from
`database.py` — a stand-in that returns realistic manufacturing metadata (material, process,
route, cost) for each part.

Then we create a `ContextPredictor`. It needs to know how to combine values from the
neighbors: prices as a weighted average, and everything else by a majority vote.


In [ ]:
import database
from hoops_ai.ml.context_layer import ContextPredictor, CategoricalRule, NumericWeightedRule

provider = database.OnDemandContextProvider()

predictor = ContextPredictor(
    provider,
    default_categorical_rule=CategoricalRule(),                 # majority vote for text fields
    per_key_rules={
        # Gears use several close grades, so we relax the winning margin a little
        # (min_margin) and tidy legacy codes (1.7147 -> 20MnCr5) before voting.
        "Material": CategoricalRule(min_margin=0.05, normalize=database.canonical_material),
        "Cost": NumericWeightedRule(log_scale=True),            # weighted average for prices
    },
)


## 3. Find parts that look like our gear

We take one gear and search the index for its 15 closest matches.


In [ ]:
gear = str(cad_files[1])
neighbors = searcher.search_by_shape(gear, top_k=15, filters={"kind": "part"})

viewer.show_search_results(neighbors, query_file=gear, grid_cols=5)


Here is the metadata already stored for those look-alike parts. Some rows are incomplete — just
like a real data store where not every part is fully tagged.


In [ ]:
from hoops_ai.insights import hits_table

known = provider.get_contexts([h.id for h in neighbors[0]])
short_neighbors, short_known = database.display_hits(neighbors[0], known)
hits_table(short_neighbors, short_known, keys=["PartFamily", "Material", "Process", "HeatTreatment", "InternalFeatures", "Cost"])


## 4. Predict the missing fields

Now we ask the predictor to fill in the fields for our gear, using the neighbors above.


In [ ]:
from hoops_ai.insights import predictions_table

prediction = predictor.infer(
    neighbors[0],
    keys=["PartFamily", "Material", "Process", "HeatTreatment", "ProcessRoute", "Cost"],
)
predictions_table(prediction)


## How to read the results

Each predicted field comes with a **confidence** and a **status**:

| Status | Meaning | What to do |
|---|---|---|
| `ready_to_propose` | The similar parts strongly agree | Safe to apply automatically |
| `needs_review` | A likely value, but not certain | Show it to an engineer to confirm |
| `insufficient_evidence` | The similar parts disagree or lack data | Leave blank; don't guess |

For this gear, the **part family** and the **manufacturing route** come back with high
confidence, while the exact material grade and cost are marked for review — the honest result
when similar parts use a few different grades.


One of the most useful predictions is the **manufacturing route** — the ordered steps to make
the part. Here it is, recovered from geometry alone:


In [ ]:
family = prediction["PartFamily"].value
route = prediction["ProcessRoute"].value

print(f"Predicted manufacturing route for this {family.lower()}:\n")
for step_number, step in enumerate(route.split(" -> "), start=1):
    print(f"  {step_number}. {step}")


## Good to know

- The predictions **reuse the metadata you already store**. The more complete and consistent
  your data, the better the results.
- The **cost is an estimate** based on similar parts, not a quote — read it as a range.
- The feature counts and costs in this demo come from an example dataset; connect your own data
  for real values.

## Use it with your own data

Replace `OnDemandContextProvider` with a small class that reads your PLM or database (implement
`get_contexts`). To control how values are combined, tidy up messy tags, tune confidence,
improve cost estimates, or find mislabeled parts, continue with
[`demo_HOOPS_Embeddings_Context_Layer_advanced.ipynb`](./demo_HOOPS_Embeddings_Context_Layer_advanced.ipynb).

**API reference:** [hoops_ai.ml.context_layer](https://docs.techsoft3d.com/hoops/ai/api_ref/hoops_ai.ml.context_layer.html)
